# Taller SQL · Soluciones de los ejercicios

Mismo dataset y función `query()` que en el cuaderno del taller. Aquí solo se muestran las **soluciones** de los 10 ejercicios.

In [ ]:
import pandas as pd
import sqlite3

CSV_EJEMPLO = '''hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,customer_type,adr,reservation_status,reservation_status_date
Resort Hotel,0,342,2015,July,1,0,0,2,0,0,BB,PRT,Transient,0.0,Check-Out,2015-07-01
Resort Hotel,0,737,2015,July,1,0,0,2,0,0,BB,GBR,Transient,0.0,Check-Out,2015-07-01
Resort Hotel,0,7,2015,July,1,0,1,1,0,0,BB,GBR,Transient,75.0,Check-Out,2015-07-02
Resort Hotel,0,13,2015,July,2,0,1,1,0,0,BB,GBR,Transient,75.0,Check-Out,2015-07-02
Resort Hotel,0,14,2015,July,3,0,2,2,0,0,BB,GBR,Transient,98.0,Check-Out,2015-07-03
City Hotel,1,2,2015,July,1,0,2,2,0,0,BB,PRT,Transient,100.0,Canceled,2015-06-29
City Hotel,0,50,2015,July,5,1,3,2,1,0,HB,ESP,Transient,120.0,Check-Out,2015-07-08
Resort Hotel,0,100,2015,August,10,1,2,2,0,0,BB,FRA,Transient,85.0,Check-Out,2015-08-12
City Hotel,1,30,2015,August,15,0,1,1,0,0,BB,DEU,Transient,60.0,Canceled,2015-08-01
Resort Hotel,0,0,2016,January,2,1,0,2,0,0,BB,PRT,Transient,90.0,Check-Out,2016-01-02
City Hotel,0,45,2016,March,14,0,2,2,0,0,SC,ITA,Transient,110.0,Check-Out,2016-03-16
Resort Hotel,1,120,2016,July,20,2,3,4,2,0,HB,GBR,Group,200.0,Canceled,2016-06-15
City Hotel,0,3,2016,December,31,1,1,2,0,0,BB,PRT,Transient,95.0,Check-Out,2017-01-01
Resort Hotel,0,21,2017,August,5,0,4,2,0,0,BB,ESP,Transient,88.0,Check-Out,2017-08-09
City Hotel,0,7,2017,August,12,1,2,2,0,0,BB,FRA,Transient,102.0,Check-Out,2017-08-14
Resort Hotel,0,14,2015,July,8,0,1,1,0,0,BB,PRT,Transient,70.0,Check-Out,2015-07-09
City Hotel,0,35,2016,February,20,0,2,2,0,0,BB,GBR,Transient,115.0,Check-Out,2016-02-22
Resort Hotel,1,90,2016,May,10,0,2,2,0,0,HB,DEU,Transient,130.0,Canceled,2016-04-15
City Hotel,0,1,2017,January,1,0,2,2,0,0,BB,PRT,Transient,99.0,Check-Out,2017-01-03
Resort Hotel,0,60,2015,September,3,1,1,2,0,0,BB,ITA,Transient,92.0,Check-Out,2015-09-04
City Hotel,0,28,2016,November,11,0,3,2,1,0,BB,ESP,Transient,105.0,Check-Out,2016-11-14
'''

from io import StringIO
df = pd.read_csv(StringIO(CSV_EJEMPLO))
conn = sqlite3.connect(':memory:')
df.to_sql('bookings', conn, index=False, if_exists='replace')

def query(sql):
    return pd.read_sql_query(sql, conn)

print('Dataset cargado. Tabla "bookings" lista.')

---
## Soluciones

### Ejercicio 1
Listar hotel, mes de llegada y ADR de las reservas **no canceladas**.

In [ ]:
query('''
  SELECT hotel, arrival_date_month, adr
  FROM bookings
  WHERE is_canceled = 0
''')

### Ejercicio 2
Contar cuántas reservas hay por `reservation_status`.

In [ ]:
query('''
  SELECT reservation_status, COUNT(*) AS num_reservas
  FROM bookings
  GROUP BY reservation_status
''')

### Ejercicio 3
Número de reservas por `arrival_date_year`, ordenado por año.

In [ ]:
query('''
  SELECT arrival_date_year, COUNT(*) AS num_reservas
  FROM bookings
  GROUP BY arrival_date_year
  ORDER BY arrival_date_year
''')

### Ejercicio 4
Suma total de `adr` por hotel (proxy de ingresos).

In [ ]:
query('''
  SELECT hotel, SUM(adr) AS total_adr
  FROM bookings
  GROUP BY hotel
''')

### Ejercicio 5
Reservas con `adr` > 100 y del **City Hotel**.

In [ ]:
query('''
  SELECT * FROM bookings
  WHERE adr > 100 AND hotel = 'City Hotel'
''')

### Ejercicio 6
Contar reservas por `customer_type`, ordenado de mayor a menor.

In [ ]:
query('''
  SELECT customer_type, COUNT(*) AS num_reservas
  FROM bookings
  GROUP BY customer_type
  ORDER BY num_reservas DESC
''')

### Ejercicio 7
Reservas cuya fecha de estado (`reservation_status_date`) sea de **2016**.

In [ ]:
query('''
  SELECT * FROM bookings
  WHERE reservation_status_date >= '2016-01-01'
    AND reservation_status_date < '2017-01-01'
''')

### Ejercicio 8
Número de reservas por mes de llegada solo para **Resort Hotel**.

In [ ]:
query('''
  SELECT arrival_date_month, COUNT(*) AS num_reservas
  FROM bookings
  WHERE hotel = 'Resort Hotel'
  GROUP BY arrival_date_month
''')

### Ejercicio 9
Países con **más de 1 reserva**, ordenados por número de reservas descendente.

*(Aquí usamos `HAVING` para filtrar después del `GROUP BY`.)*

In [ ]:
query('''
  SELECT country, COUNT(*) AS num_reservas
  FROM bookings
  GROUP BY country
  HAVING COUNT(*) > 1
  ORDER BY num_reservas DESC
''')

### Ejercicio 10
Reservas de **julio o agosto**, **no canceladas**; mostrar hotel, mes y adr, ordenado por adr descendente.

In [ ]:
query('''
  SELECT hotel, arrival_date_month, adr
  FROM bookings
  WHERE arrival_date_month IN ('July', 'August')
    AND is_canceled = 0
  ORDER BY adr DESC
''')